# vesicletrack — walkthrough

Detect, track and score vesicles in a time-lapse movie.

Run top to bottom. It works on a synthetic movie with **known ground truth**, so
you can see what a correct result looks like before pointing it at your own data.
Swap the path in section 2 when you are ready.

## 1. Setup

If the package is not installed (`pip install -e .`), the cell below adds `src/`
to the path so the notebook works from a fresh clone.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
root = root if (root / 'src').exists() else root.parent
sys.path.insert(0, str(root / 'src'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from vesicletrack import Config, analyse, analyse_many, metrics, render
print('vesicletrack ready, working from', root)

## 2. Make (or point at) a movie

The synthetic movie has 40 static vesicles, 6 movers travelling 14 px, and 3 px of
stage drift. Replace `MOVIE` with your own `.tif` / `.nd2` when ready.

The stack must be **(T, Y, X)**. If yours has a channel or z axis, pass
`channel=` or `z_project='max'` to `analyse` — it will not guess which axis is time.

In [ ]:
import subprocess
MOVIE = root / 'examples' / 'synthetic.tif'
if not MOVIE.exists():
    subprocess.run([sys.executable, str(root/'examples'/'make_synthetic.py'),
                    str(MOVIE)], check=True)
print(MOVIE)

## 3. Parameters

Everything the pipeline does lives in the config. Two fields **must** be right for
your data and cannot be inferred from the images:

- `dt_seconds` — every rate scales through it
- `um_per_px` — leave `None` to stay in pixels

Set `dt_seconds` wrong and every rate is wrong while nothing *looks* wrong.

In [ ]:
cfg = Config.load(root / 'config' / 'default.yaml')

# synthetic movie: 20 fps, and it is shorter than the real recordings,
# so the coarse-graining windows are scaled down to match.
cfg.set('dt_seconds', 0.05)
cfg.set('um_per_px', 0.107)
cfg.set('metrics.tau_frames', 10)            # ~0.5 s
cfg.set('metrics.tau_directed_frames', 40)   # ~2 s
cfg.set('link.min_length_frames', 30)
cfg.set('output_dir', str(root / 'output'))   # absolute, so it lands here wherever the notebook is run from
cfg.validate()

print(f'tau        = {cfg.tau_seconds:.2f} s')
print(f'tau_direct = {cfg.tau_directed_seconds:.2f} s')

## 4. Run

load → drift-correct → detect → link → score → classify.

Drift correction runs **first**: stage drift moves every vesicle together, so
uncorrected it reads as directed transport in all of them at once.

In [ ]:
res = analyse(MOVIE, cfg, name='demo')
res.counts()

### Check the invariant

`gross ≥ directed ≥ net_coarse` is guaranteed by the triangle inequality for every
vesicle. If this is not empty, something upstream is wrong — duplicated frames,
unsorted tracks, NaNs — not the metric.

In [ ]:
bad = res.check()
print('ordering violations:', len(bad))
bad.head()

## 5. Look at the result

The three panels answer the two questions worth asking before trusting a number:
did detection **find** the vesicles, and did classification **label** them sensibly.

In [ ]:
out = res.save()
for k, v in out.items():
    print(f'{k:16s} {v}')

In [ ]:
from IPython.display import Image, display
display(Image(str(out['three_panel'])))

### The three distances

For the synthetic data the truth is known: the movers travelled 14 px ≈ 1.5 µm.
Watch what **gross** reports for the same vesicles.

In [ ]:
v = res.vesicles
cols = ['net_um','directed_um','gross_um','runs_p']
v.groupby('klass')[cols].median().round(3)

In [ ]:
display(Image(str(out['distances'])))

A single vesicle, with all three distances drawn on the same axes — the quickest
way to see why they differ so much.

In [ ]:
mv = res.movers().iloc[0]
p = root / 'output' / 'demo' / 'vesicles' / f"vesicle_{int(mv.particle):04d}.png"
display(Image(str(p))) if p.exists() else print('enable render.per_vesicle_images')

## 6. Tuning

`cfg.copy(**overrides)` makes a modified config without touching the original, so
a sweep is a loop rather than a series of edits.

Detection threshold is usually the first thing to set: too low invents spots, too
high loses dim vesicles.

In [ ]:
rows = []
for thr in [2.5, 3.0, 3.5, 4.0]:
    r = analyse(MOVIE, cfg.copy(**{'detect.threshold_sigma': thr}),
                name=f'thr{thr}', verbose=False)
    c = r.counts()
    rows.append(dict(threshold=thr, vesicles=r.summary['n_vesicles'],
                     movers=int(c.get('mover', 0)),
                     confined=int(c.get('confined', 0)),
                     excluded=int(c.get('excluded', 0))))
pd.DataFrame(rows)

Truth is 6 movers out of 46 vesicles. A threshold that changes the **mover** count
a lot is changing your answer, not just your detection sensitivity — worth knowing
before picking one.

### Does τ matter?

τ sets what counts as "consistent direction". Too small and localisation noise
survives; too large and genuine reversals are erased.

In [ ]:
rows = []
for tau in [10, 20, 40, 80]:
    r = analyse(MOVIE, cfg.copy(**{'metrics.tau_directed_frames': tau}),
                name=f'tau{tau}', verbose=False)
    g = r.vesicles.groupby('klass').directed.median()
    rows.append(dict(tau_frames=tau, tau_s=tau*cfg.dt_seconds,
                     mover=g.get('mover', np.nan),
                     confined=g.get('confined', np.nan)))
d = pd.DataFrame(rows)
d['separation'] = d.mover / d.confined
d.round(2)

The ratio is the signal-to-noise of the metric: how far the movers sit above the
confined population. It should rise with τ and then flatten.

## 7. Videos

Off by default — per-vesicle videos are one file each. `ffmpeg` is found on `PATH`
or next to the running Python; if it is missing you get a warning, not a crash.

In [ ]:
res_v = analyse(MOVIE, cfg.copy(**{'render.per_vesicle_videos': True,
                                  'render.overview_video': True,
                                  'render.max_vesicle_outputs': 3}),
                name='demo_video', verbose=False)
for k, v in res_v.save().items():
    print(f'{k:16s} {v}')

## 8. Every vesicle is reported — filters only *label*

`res.vesicles` holds **every** tracked vesicle with `passes_filter` and a
`filter_reason`. `res.filtered` is the passing subset. Both are written per movie.

This matters: once a vesicle is deleted upstream you cannot ask how many were
excluded, whether they differed systematically, or what another threshold would
have given — and those are the questions a reviewer asks.

In [ ]:
print(f'all      {len(res.vesicles)}')
print(f'passing  {len(res.filtered)}')
fails = res.vesicles[~res.vesicles.passes_filter]
fails.filter_reason.value_counts() if len(fails) else 'every vesicle passed'


## 9. Lifetime, gaps and censoring

**`span_frames` and `observed_frames` are different numbers** whenever the linker
bridged a dropout. Reporting only span describes a vesicle as present in frames
where nothing was detected.

In [ ]:
cols = ['span_frames','observed_frames','frac_observed','n_gaps',
        'longest_gap','is_censored']
res.vesicles[cols].describe().loc[['min','50%','max']].round(2)

`is_censored` marks vesicles present in the first or last frame — their true
lifetime was never observed. Pooling them with complete observations biases mean
lifetime **downward**, so check this before quoting one.

In [ ]:
v = res.vesicles
complete = v[~v.is_censored]
print(f"censored: {int(v.is_censored.sum())}/{len(v)}")
if len(complete):
    print('mean span, complete only :', round(complete.span_s.mean(), 2), 's')
    print('mean span, all pooled    :', round(v.span_s.mean(), 2), 's  <- biased low')
else:
    print('EVERY vesicle is censored: they are all present in the first and last\n'
          'frame, so no complete lifetime was observed and a mean lifetime cannot\n'
          'be quoted from this movie at all. On the synthetic fixture that is by\n'
          'construction; on real data it is common for long-lived vesicles and is\n'
          'exactly what is_censored exists to make visible.')


## 10. Size

`sigma_px` is what was measured and **includes the PSF**; `sigma_deconv_px` is the
excess over it. A vesicle is below the diffraction limit, so treat ~0.4 px of
deconvolved width as indistinguishable from zero — synthetic point sources with a
true size of *zero* come back at 0.41 px.

For comparing conditions imaged identically, prefer raw `sigma_px`: the PSF
contribution is common to both and cancels.

In [ ]:
res.vesicles[['sigma_px','sigma_deconv_px','fwhm_px','at_diffraction_limit']].describe().round(3)

## 11. ROIs — in and out

Accepts ImageJ `.roi`/`.zip`, mask or label images, arrays, or named polygons.
Vesicles outside every region are **kept** and labelled `outside`, because outside
is usually the comparison group.

In [ ]:
import numpy as np
H, W = res.stack.shape[1:]
rois = {'left_half':  [(0,0), (W//2,0), (W//2,H), (0,H)],
        'right_half': [(W//2,0), (W,0), (W,H), (W//2,H)]}
r_roi = analyse(MOVIE, cfg, name='roi_demo', rois=rois, verbose=False)
r_roi.vesicles.groupby('roi')[['net','sigma_px']].agg(['count','median']).round(3)

`roi_changed` flags vesicles that visited more than one region; `roi_frac` is how
much of the track was in the assigned one, so values near 0.5 straddle a boundary.

In [ ]:
r_roi.vesicles[['roi','roi_frac','roi_changed','n_rois_visited']].head()

## 12. Two metric families — do not mix them up

- `directed` = L(`tau_directed_frames`), the coarse path
- `runs_total` = run detection on coarse steps at the **shorter** `tau_frames`

**`runs_p` tests `runs_total`, not `directed`.** These were once named
`runs_p` / `directed_runs` and sat beside `directed` in the output, which
invited "directed displacement was significant (p<0.05)" written about a different
number. Both τ values are columns so the file is self-describing.

`directed` is **NaN**, never 0, for tracks spanning under two τ-windows —
`directed_measurable` says which.

In [ ]:
v = res.vesicles
print(f"measurable: {int(v.directed_measurable.sum())}/{len(v)}")
print(f"directed == 0 exactly: {int((v.directed == 0).sum())}   (must be 0)")
v[['directed','directed_measurable','runs_total','runs_p','runs_z',
   'tau_frames','tau_directed_frames']].head()

## 13. Batch, and CSVs at every level

Metadata comes from a **sample sheet**, not from filename parsing — your naming
scheme stays yours and out of the package.

In [ ]:
from vesicletrack import aggregate

runs = []
for nm, geno in [('c1','WT'), ('c2','KO')]:
    runs.append(analyse(MOVIE, cfg, name=nm, verbose=False, genotype=geno))

written = aggregate.write_all(runs, root/'output'/'batch', by='genotype')
for k, p in written.items():
    print(f'{k:20s} {p}')

In [ ]:
pd.read_csv(written['per_video'])[
    ['video','genotype','n_vesicles_all','n_vesicles','net_median','sigma_px_median']
].round(3)

**`n` at group level counts VIDEOS, not vesicles.** Vesicles within one cell are
not independent; treating each as a replicate inflates `n` by hundreds and produces
significance that will not survive a nested analysis. `sem` is across video means.

In [ ]:
pd.read_csv(written['per_genotype'])[
    ['genotype','n_videos','n_vesicles_total','net_median_mean','net_median_sem']
].round(3)

### From the shell

```bash
vesicletrack "data/*.tif" --dt 0.0446 --um-per-px 0.107 \
    --sheet samples.csv --by genotype --rois cell_mask.tif -o output
```

Writes the per-movie folders *and* every aggregate CSV in one command.

## 14. Reproducibility

Every run writes `config_used.yaml` beside its outputs, so any figure traces back
to the exact parameters that produced it.

In [ ]:
print((root/'output'/'demo'/'config_used.yaml').read_text()[:400])